<h3><b>Implement Gradient Checking</b></h3>

In [1]:
import numpy as np

In [57]:
def dictionary_to_vector(parameters):
    """
    Roll all our parameters dictionary into a single vector satisfying our specific required shape.
    """
    keys = []
    count = 0
    params = list(parameters.keys())
    
    for key in params: 

        new_vector = np.reshape(parameters[key], (-1, 1))
        keys.append((key, new_vector.shape[0], parameters[key].shape))
        
        if count == 0:
            theta = new_vector
        else:
            theta = np.concatenate((theta, new_vector), axis=0)
        count = count + 1

    return theta, keys

In [58]:
h = {"W1": np.array([23, 23, 23]).reshape((3, 1)), "b1": np.array([23, 34, 21]).reshape((3, 1))}
print(h)
t, k = dictionary_to_vector(h)
print(t)
print(k)
print(t.shape)


{'W1': array([[23],
       [23],
       [23]]), 'b1': array([[23],
       [34],
       [21]])}
[[23]
 [23]
 [23]
 [23]
 [34]
 [21]]
[('W1', 3, (3, 1)), ('b1', 3, (3, 1))]
(6, 1)


In [59]:
def vector_to_dictionary(theta, keys):
    """
    Unroll all our parameters dictionary from a single vector satisfying our specific required shape.
    """
    parameters = {}
    count = 0
    for key in keys:
        
        parameters[key[0]] = theta[count: count + key[1]].reshape(key[-1])
        count += key[1]
        
    return parameters

In [60]:
p = vector_to_dictionary(t, k)
p

{'W1': array([[23],
        [23],
        [23]]),
 'b1': array([[23],
        [34],
        [21]])}

In [71]:
def gradients_to_vector(gradients):
    """
    Roll all our gradients dictionary into a single vector satisfying our specific required shape.
    """
    
    count = 0

    params = list(gradients.keys())
    
    for key in params: 

        new_vector = np.reshape(gradients[key], (-1, 1))

        if count == 0:
            theta = new_vector
        else:
            theta = np.concatenate((theta, new_vector), axis=0)
        count = count + 1

    return theta

In [72]:
h = {"dW1": np.array([23, 23, 23]).reshape((3, 1)), "db1": np.array([23, 34, 21]).reshape((3, 1))}
print(h)
t = gradients_to_vector(h)
print(t)
print(t.shape)

{'dW1': array([[23],
       [23],
       [23]]), 'db1': array([[23],
       [34],
       [21]])}
[[23]
 [23]
 [23]
 [23]
 [34]
 [21]]
(6, 1)


In [100]:
def sigmoid(Z):

        A = 1 / (1 + np.exp(-Z))
        cache = Z

        return A, cache

def relu(Z):

    A = np.maximum(0, Z)
    cache = Z

    return A, cache


def forward_propagation(X, Y, parameters, hidden_activation, output_activation):

        caches = []
        A = X
        L = len(parameters) // 2  # number of layers in the network

        for l in range(1, L):

            A_prev = A
            W = parameters[f"W{l}"]
            b = parameters[f"b{l}"]

            if hidden_activation == "relu":

                Z = np.dot(W, A_prev) + b
                linear_cache = (A_prev, W, b)
                A, activation_cache = relu(Z=Z)

                caches.append((linear_cache, activation_cache))
            else:
                raise ValueError("hidden_activation must be 'relu'!")

        A_prev = A
        W = parameters[f"W{L}"]
        b = parameters[f"b{L}"]

        if output_activation == "sigmoid":

            Z = np.dot(W, A_prev) + b
            linear_cache = (A_prev, W, b)
            A, activation_cache = sigmoid(Z=Z)

            caches.append((linear_cache, activation_cache))
        else:
            raise ValueError("output_activation must be 'sigmoid'!")

        cost = - np.sum(np.multiply(Y, np.log(A)) + np.multiply(1 - Y, np.log(1 - A))) / Y.shape[1]

        return A, cost, caches

In [105]:
def derivative_sigmoid(dA, cache):

        Z = cache
        A = 1 / (1 + np.exp(-Z))
        dZ = dA * A * (1 - A)

        assert (dZ.shape == Z.shape)

        return dZ

def derivative_relu(dA, cache):

    Z = cache
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0

    assert (dZ.shape == Z.shape)

    return dZ

def backward_propagation(A, Y, caches, hidden_activation, output_activation, dataset):

        grads = {}
        L = len(caches)  # number of network layers
        Y = Y.reshape(A.shape)

        dA = - (np.divide(Y, A ) - np.divide(1 - Y, 1 - A))

        if output_activation == "sigmoid":

            current_cache = caches[-1]
            (linear_cache, activation_cache) = current_cache
            (A_prev, W, b) = linear_cache

            dZ = derivative_sigmoid(dA=dA, cache=activation_cache)

            m = A_prev.shape[1]
            if dataset == "train":

                dW = 1. / m * np.dot(dZ, A_prev.T)
                db = np.sum(dZ, axis=1, keepdims=True) / m
                dA_prev = np.dot(W.T, dZ)

            else:
                dW = 1. / m * np.dot(dZ, A_prev.T)
                db = np.sum(dZ, axis=1, keepdims=True) / m
                dA_prev = np.dot(W.T, dZ)

            grads[f"dW{L}"] = dW
            grads[f"db{L}"] = db
            grads[f"dA{L-1}"] = dA_prev

        else:
            raise ValueError("output_activation must be 'sigmoid'!")

        for l in reversed(range(L - 1)):

            if hidden_activation == "relu":

                current_cache = caches[l]
                (linear_cache, activation_cache) = current_cache
                (A_prev, W, b) = linear_cache
                dZ = derivative_relu(dA=grads[f"dA{l+1}"], cache=activation_cache)

                m = A_prev.shape[1]
                if dataset == "train":

                    dW = 1. / m * np.dot(dZ, A_prev.T)
                    db = np.sum(dZ, axis=1, keepdims=True) / m
                    dA_prev = np.dot(W.T, dZ)

                else:

                    dW = 1. / m * np.dot(dZ, A_prev.T)
                    db = np.sum(dZ, axis=1, keepdims=True) / m
                    dA_prev = np.dot(W.T, dZ)

                grads[f"dW{l+1}"] = dW
                grads[f"db{l+1}"] = db
                grads[f"dA{l}"] = dA_prev

            else:
                raise ValueError("hidden_activation must be 'relu'!")

        return grads

In [130]:
def gradient_check(parameters, gradients, X, Y, epsilon=1e-7, trigger=2e-7, print_msg=False):
    """
    Checks if backward_propagation computes correctly the gradient of the cost output by forward_propagation

    """
    
    parameters_values, keys = dictionary_to_vector(parameters)
    grad = gradients_to_vector(gradients)
    num_parameters = parameters_values.shape[0]
    
    J_plus = np.zeros((num_parameters, 1))
    J_minus = np.zeros((num_parameters, 1))
    gradapprox = np.zeros((num_parameters, 1))
    
    
    for i in range(num_parameters):
        
        theta_plus = np.copy(parameters_values)
        theta_plus[i] = theta_plus[i] + epsilon
        _, J_plus[i], _ = forward_propagation(X, Y, vector_to_dictionary(theta_plus, keys), "relu", "sigmoid")
        
        
        theta_minus = np.copy(parameters_values)
        theta_minus[i] = theta_minus[i] - epsilon
        _, J_minus[i], _ = forward_propagation(X, Y, vector_to_dictionary(theta_minus, keys), "relu", "sigmoid")
          
        gradapprox[i] = (J_plus[i] - J_minus[i]) / (2 * epsilon)
        
    print("Shape of gradapprox:", gradapprox.shape)
    print("Shape of grad:", grad.shape)
    print("Number of parameters:", grad.shape[0])
    print("Number of gradients:", grad.shape[0])

    numerator = np.linalg.norm(gradapprox - grad)
    denominator = np.linalg.norm(gradapprox) + np.linalg.norm(grad)
    difference = numerator / denominator
    
    if print_msg:
        if difference > trigger:
            print ("\033[93m" + "There is a mistake in the backward propagation! difference = " + str(difference) + "\033[0m")
        else:
            print ("\033[92m" + "Your backward propagation works perfectly fine! difference = " + str(difference) + "\033[0m")

    return difference

In [131]:
np.random.seed(1)
X = np.random.randn(4,3)
Y = np.array([1, 1, 0]).reshape((3, 1))
W1 = np.random.randn(5,4) 
b1 = np.random.randn(5,1) 
W2 = np.random.randn(3,5) 
b2 = np.random.randn(3,1) 
W3 = np.random.randn(1,3) 
b3 = np.random.randn(1,1) 
parameters = {"W1": W1,
              "b1": b1,
              "W2": W2,
              "b2": b2,
              "W3": W3,
              "b3": b3}

In [143]:
A, cost, caches = forward_propagation(X, Y, parameters, "relu", "sigmoid")
print(cost)
print(A.shape)
print(len(caches))

12.334852484246294
(1, 3)
3


In [140]:
grads = backward_propagation(A, Y, caches, "relu", "sigmoid", "train")
print(grads.keys())
del grads["dA0"]
print(grads.keys())


dict_keys(['dW3', 'db3', 'dA2', 'dW2', 'db2', 'dA1', 'dW1', 'db1', 'dA0'])
dict_keys(['dW3', 'db3', 'dA2', 'dW2', 'db2', 'dA1', 'dW1', 'db1'])


In [144]:
differences = gradient_check(parameters=parameters, gradients=grads, X=X, Y=Y, epsilon=1e-7, trigger=2e-7, print_msg=True)